# Decomposing wealth and residential inequalities in facility delivery in Tanzania

Analysis code for a **Blinder–Oaxaca-type decomposition** of rural–urban and
richest–poorest inequalities in health facility delivery among women in Tanzania,
using the **2022 TDHS-MIS** women's individual recode (`TZIR81FL.DTA`).

**Author:** Clifford Silver Tarimo · Dar es Salaam Institute of Technology

**Data:** Not included in this repository. Freely available from the DHS Program
(https://dhsprogram.com/data/) after registration. Download the Tanzania 2022
*Individual Recode* in Stata format (`TZIR81FL.DTA`) and place it beside this
notebook, or edit `DATA_DIR` below.

This notebook reproduces the descriptive prevalence estimates, the two decomposition
analyses, the wealth-by-residence interaction, and the ANC 8+ sensitivity analysis.

## 1. Environment

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

print("Python    :", sys.version.split()[0])
print("pandas    :", pd.__version__)
print("numpy     :", np.__version__)
print("matplotlib:", matplotlib.__version__)

## 2. Load data

Set `DATA_DIR` to the folder that contains `TZIR81FL.DTA`. By default it looks in
the same folder as this notebook.

In [ ]:
from pathlib import Path

DATA_DIR = Path(".")               # folder containing TZIR81FL.DTA
IR_FILE  = DATA_DIR / "TZIR81FL.DTA"
assert IR_FILE.exists(), f"TZIR81FL.DTA not found. Place it in {DATA_DIR.resolve()}"

cols = [
    "caseid",
    "v005",    # women's sample weight
    "v021",    # primary sampling unit (cluster)
    "v022",    # sampling stratum
    "v023",    # sampling stratum (alternative)
    "v024",    # region
    "v025",    # type of residence (urban/rural)
    "v106",    # maternal education
    "v190",    # household wealth quintile
    "v467d",   # distance to health facility is a big problem
    "m14_1",   # number of ANC visits, most recent birth
    "m15_1",   # place of delivery, most recent birth
    "b19_01",  # age of most recent child (months)
]
df = pd.read_stata(IR_FILE, columns=cols, convert_categoricals=True)
df["weight"] = df["v005"] / 1_000_000
print("Rows in individual recode:", len(df))

## 3. Outcome and analytic sample

The outcome is **facility delivery** for each woman's most recent live birth
(`m15_1`). The analytic sample is women with non-missing place of delivery.

In [ ]:
analysis = df[df["m15_1"].notna()].copy()

home_categories = [
    "respondent's home", "other home", "tba premises",
    "on the way to the hospital", "other",
]
analysis["facility_delivery"] = np.where(
    analysis["m15_1"].isin(home_categories), 0, 1
)
print("Analytic sample (recent birth, place of delivery known):", len(analysis))

## 4. Derive covariates

In [ ]:
analysis["residence"] = analysis["v025"].astype(str)
analysis["education"] = analysis["v106"].astype(str)
analysis["wealth"]    = analysis["v190"].astype(str)
analysis["region"]    = analysis["v024"].astype(str)

# ANC visits: "no antenatal visits" -> 0 ; "don't know" -> missing
analysis["anc_visits"] = pd.to_numeric(analysis["m14_1"], errors="coerce")
analysis.loc[analysis["m14_1"].astype(str) == "no antenatal visits", "anc_visits"] = 0
analysis["anc_4plus"] = np.where(analysis["anc_visits"].isna(), np.nan,
                                 np.where(analysis["anc_visits"] >= 4, 1, 0))
analysis["anc_8plus"] = np.where(analysis["anc_visits"].isna(), np.nan,
                                 np.where(analysis["anc_visits"] >= 8, 1, 0))

# Perceived distance: big problem (1) vs not a big problem (0)
analysis["distance_big_problem"] = np.where(
    analysis["v467d"].astype(str) == "big problem", 1, 0
)

# Ordered categoricals so reference groups are meaningful
analysis["wealth"] = pd.Categorical(
    analysis["wealth"], ["poorest","poorer","middle","richer","richest"], ordered=True)
analysis["education"] = pd.Categorical(
    analysis["education"], ["no education","primary","secondary","higher"], ordered=True)

## 5. Weighted descriptive prevalence

Weighted facility-delivery prevalence overall and by subgroup.

In [ ]:
def wmean(x, w):
    return np.average(x, weights=w)

def weighted_prevalence_by(group_var, data=analysis):
    d = data.dropna(subset=[group_var, "facility_delivery", "weight"])
    out = (d.groupby(group_var, observed=True)
             .apply(lambda x: wmean(x["facility_delivery"], x["weight"]) * 100,
                    include_groups=False)
             .rename("facility_delivery_%")
             .reset_index())
    return out

print("Overall weighted facility delivery: %.1f%%" %
      (wmean(analysis["facility_delivery"], analysis["weight"]) * 100))

for v in ["residence", "wealth", "education", "anc_4plus", "distance_big_problem", "region"]:
    print("\n", v.upper())
    print(weighted_prevalence_by(v).to_string(index=False))

## 6. Blinder–Oaxaca-type decomposition

The explained (composition) component is estimated from a **pooled weighted linear
probability model**: `E = Σ (x̄_hi − x̄_lo) · β̂`. The unexplained component is the
remainder `U = Δ − E`. Contributions are also aggregated by covariate domain.

In [ ]:
def decompose(data, group_col, group_hi, predictors,
              outcome="facility_delivery", weight="weight"):
    """Blinder-Oaxaca-type explained-component decomposition (pooled WLS)."""
    d = data.dropna(subset=[outcome, weight, group_col] + predictors).copy()
    X = pd.get_dummies(d[predictors], drop_first=True).astype(float)
    y = d[outcome].astype(float).values
    w = d[weight].astype(float).values
    hi = (d[group_col] == group_hi).values

    # Weighted least squares on the pooled sample
    Xi = X.copy(); Xi.insert(0, "intercept", 1.0)
    sw = np.sqrt(w)
    beta = np.linalg.lstsq(Xi.values * sw[:, None], y * sw, rcond=None)[0]
    coef = pd.Series(beta, index=Xi.columns)

    # Weighted covariate means by group and contributions
    m_hi = np.average(X[hi], axis=0, weights=w[hi])
    m_lo = np.average(X[~hi], axis=0, weights=w[~hi])
    contrib = (m_hi - m_lo) * coef[X.columns].values

    r_hi = np.average(y[hi], weights=w[hi])
    r_lo = np.average(y[~hi], weights=w[~hi])
    gap = r_hi - r_lo
    explained = contrib.sum()
    unexplained = gap - explained

    res = pd.DataFrame({
        "predictor": X.columns,
        "contribution_pp": contrib * 100,
        "percent_of_gap": contrib / gap * 100,
    })
    return res, gap, explained, unexplained

def domain(p):
    if p.startswith("wealth"):    return "Wealth"
    if p.startswith("education"): return "Education"
    if p.startswith("residence"):return "Urban residence"
    if p.startswith("anc"):       return "ANC visits"
    if p.startswith("distance"):  return "Distance problem"
    return p

def summarise(res, gap, explained, unexplained, title):
    res = res.copy(); res["domain"] = res["predictor"].apply(domain)
    dom = (res.groupby("domain")[["contribution_pp", "percent_of_gap"]]
              .sum().sort_values("contribution_pp", ascending=False))
    print(f"=== {title} ===")
    print(f"Gap: {gap*100:.1f} pp | Explained: {explained*100:.1f} pp "
          f"({explained/gap*100:.1f}%) | Unexplained: {unexplained*100:.1f} pp "
          f"({unexplained/gap*100:.1f}%)")
    print(dom.round(1).to_string())
    print()

### 6a. Rural–urban decomposition

In [ ]:
res_ru, gap_ru, exp_ru, unexp_ru = decompose(
    analysis, "residence", "urban",
    ["wealth", "education", "anc_4plus", "distance_big_problem"])
summarise(res_ru, gap_ru, exp_ru, unexp_ru, "Rural-urban gap")

### 6b. Richest–poorest decomposition (wealth omitted; it defines the groups)

In [ ]:
wealth_sub = analysis[analysis["wealth"].isin(["poorest", "richest"])]
res_rp, gap_rp, exp_rp, unexp_rp = decompose(
    wealth_sub, "wealth", "richest",
    ["residence", "education", "anc_4plus", "distance_big_problem"])
summarise(res_rp, gap_rp, exp_rp, unexp_rp, "Richest-poorest gap")

## 7. Wealth-by-residence interaction (descriptive)

Weighted facility-delivery prevalence within each wealth × residence subgroup, and the
rural–urban gap within each wealth quintile.

In [ ]:
inter = (analysis
    .dropna(subset=["wealth", "residence", "facility_delivery", "weight"])
    .groupby(["wealth", "residence"], observed=True)
    .apply(lambda x: pd.Series({
        "n": len(x),
        "facility_delivery_%": wmean(x["facility_delivery"], x["weight"]) * 100,
    }), include_groups=False)
    .reset_index())

pivot = inter.pivot(index="wealth", columns="residence", values="facility_delivery_%")
pivot["urban_minus_rural_pp"] = pivot["urban"] - pivot["rural"]
print(pivot.round(1).to_string())

## 8. Sensitivity analysis: ANC 8+ instead of ANC 4+

Re-run both decompositions substituting an ANC 8+ threshold (WHO recommendation).

In [ ]:
res_ru8, gap_ru8, exp_ru8, unexp_ru8 = decompose(
    analysis, "residence", "urban",
    ["wealth", "education", "anc_8plus", "distance_big_problem"])
res_rp8, gap_rp8, exp_rp8, unexp_rp8 = decompose(
    wealth_sub, "wealth", "richest",
    ["residence", "education", "anc_8plus", "distance_big_problem"])

print("Rural-urban   : ANC4+ %.1f%%  vs  ANC8+ %.1f%% explained"
      % (exp_ru/gap_ru*100, exp_ru8/gap_ru8*100))
print("Richest-poorest: ANC4+ %.1f%%  vs  ANC8+ %.1f%% explained"
      % (exp_rp/gap_rp*100, exp_rp8/gap_rp8*100))

---
*End of analysis. Figures in the manuscript (regional map, wealth×residence
interaction plot, decomposition waterfall/sensitivity plots) are generated from the
objects above; see the manuscript for their final formatted versions.*